# Run v6 pipeline locally

**Architecture change from v5.2:** the LLM no longer extracts benefit rows from input. Python does the structural work; the LLM only generates `benefitdesc` and `tinyDescription` text per row.

**Output: 10 columns exactly** - `planid, plantypeid, benefitid, benefitname, coverageTypeid, coverageTypedesc, serviceTypeID, serviceTypeDesc, benefitdesc, tinyDescription`

**Lookups via CSV** - swap the placeholder CSVs for your real IFP2-derived ones and the pipeline picks them up. No code changes needed.

**No Azure App Service.** No Flask. Notebook calls Python directly.

## 0. Config

In [ ]:
import os
import sys
from pathlib import Path

# Make the module importable
WORK_DIR = Path.cwd()
sys.path.insert(0, str(WORK_DIR))

# ----- INPUTS -----
INPUT_CSV = WORK_DIR / 'input_MedicareFeed_carrierDESR_PBP_LoadID211.csv'

# ----- LOOKUPS (the three CSVs you'll provide) -----
LOOKUP_DIR = WORK_DIR / 'lookups'

# ----- PROMPTS -----
PROMPTS_DIR = WORK_DIR / 'prompts'

# ----- OUTPUTS -----
# JSON: the canonical deliverable for Morteza's SQL ingest ({load_id, results: [...]})
OUTPUT_JSON = WORK_DIR / 'output_benefits_211.json'
# CSV: optional side artifact for spot-checking in Excel
OUTPUT_CSV  = WORK_DIR / 'output_benefits_211.csv'

# ----- DEV: process only one plan for quick iteration -----
PLAN_FILTER = None  # e.g. ['HCSC_H0107-003-000'] for single-plan testing

print(f'Input:        {INPUT_CSV}')
print(f'Lookups:      {LOOKUP_DIR}')
print(f'Prompts:      {PROMPTS_DIR}')
print(f'Output JSON:  {OUTPUT_JSON}  (canonical for ingest)')
print(f'Output CSV:   {OUTPUT_CSV}   (spot-check artifact)')
print(f'Filter:       {PLAN_FILTER}')


## 1. Verify env vars for Azure OpenAI

In [ ]:
required_vars = ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY']
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise SystemExit(f'Missing env vars: {missing}\n'
                     f'Set them in your shell or in .env before running.')
print(f'AZURE_OPENAI_ENDPOINT:   {os.getenv("AZURE_OPENAI_ENDPOINT")[:50]}...')
print(f'AZURE_OPENAI_DEPLOYMENT: {os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")}')
print(f'AZURE_OPENAI_API_KEY:    set ({len(os.getenv("AZURE_OPENAI_API_KEY"))} chars)')


## 2. Verify input + lookups exist and are sane

Fast smoke check - reads the files and shows their shape. If any are missing, fix that before running the pipeline.

In [ ]:
import pandas as pd

# Input
inp_df = pd.read_csv(INPUT_CSV)
print(f'Input:       {len(inp_df):,} rows, {inp_df["FileName"].nunique()} plans')

# Lookups
for fn in ['benefit_id_name.csv', 'coverage_type_lookup.csv', 'service_type_lookup.csv']:
    path = LOOKUP_DIR / fn
    if not path.exists():
        print(f'MISSING: {path}')
    else:
        d = pd.read_csv(path)
        print(f'  {fn}: {len(d)} rows, columns={list(d.columns)}')


## 3. Dry run: deterministic structure only (no LLM)

Shows what rows WILL be produced before paying for LLM calls. Lets you spot-check that the right benefits are picked up for one plan.

In [ ]:
from build_benefits_v6 import (
    Lookups, _extract_plan_metadata, _group_rows_by_benefit, _build_base_rows,
)

# Pick one plan for the dry run
sample_plan_fn = inp_df['FileName'].iloc[0] if not PLAN_FILTER else PLAN_FILTER[0]
sample_rows = inp_df[inp_df['FileName'] == sample_plan_fn].copy()

lookups = Lookups.from_dir(LOOKUP_DIR)
plan_meta = _extract_plan_metadata(sample_rows)
groups = _group_rows_by_benefit(sample_rows)
base_rows = _build_base_rows(plan_meta, groups, lookups)

print(f'Plan: {plan_meta["planid"]} ({plan_meta["plantypeid"]})')
print(f'Would produce {len(base_rows)} output rows from {len(groups)} benefit groups')
print(f'\nBase row preview:')
preview_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in base_rows])
preview_df = preview_df[['planid', 'plantypeid', 'benefitid', 'benefitname',
                          'coverageTypeid', 'coverageTypedesc',
                          'serviceTypeID', 'serviceTypeDesc']]
print(preview_df.head(20).to_string())


## 4. Full run (with LLM)

Calls LLM once per base row to fill in `benefitdesc` and `tinyDescription`. For a single plan with ~60 benefits and `MAX_CONCURRENCY=16`, expect ~30-60 seconds.

In [ ]:
from build_benefits_v6 import run_load
import time

t0 = time.monotonic()
summary = run_load(
    input_csv_path   = INPUT_CSV,
    lookup_dir       = LOOKUP_DIR,
    prompts_dir      = PROMPTS_DIR,
    output_json_path = OUTPUT_JSON,
    output_csv_path  = OUTPUT_CSV,
    plan_filter      = PLAN_FILTER,
)
total = time.monotonic() - t0

print(f'\n=== Run Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')


## 5. Inspect the output

In [ ]:
out_df = pd.read_csv(OUTPUT_CSV)
print(f'Output: {len(out_df)} rows x {len(out_df.columns)} cols')
print(f'Columns: {list(out_df.columns)}')
print()
print('First 10 rows:')
print(out_df.head(10).to_string())


### Rows per benefit / coverage type / service type

In [ ]:
print('Distinct planids:    ', out_df['planid'].nunique())
print('Distinct benefitids: ', out_df['benefitid'].nunique())
print()
print('Rows per benefitid:')
print(out_df.groupby(['benefitid', 'benefitname']).size().to_string())


## 6. Compare to expected IFP2 output (if available)

Loads a reference IFP2 export and compares benefit-id coverage. Useful for spotting which benefits the section-code map is still missing.

In [ ]:
REF_PATH = WORK_DIR / 'output_ifp2_planbenefits_planID_H0107_003_endingPeriodID_336.csv'

if REF_PATH.exists():
    ref_df = pd.read_csv(REF_PATH, engine='python', on_bad_lines='skip')
    ref_bids = set(ref_df['benefitID'].astype(int).unique())
    out_bids = set(out_df['benefitid'].astype(int).unique())
    print(f'Reference IFP2: {len(ref_bids)} unique benefit IDs')
    print(f'Our v6 output:  {len(out_bids)} unique benefit IDs')
    print()
    missing = sorted(ref_bids - out_bids)
    extra   = sorted(out_bids - ref_bids)
    print(f'In reference but NOT in our output: {len(missing)}')
    for bid in missing:
        ref_row = ref_df[ref_df['benefitID'] == bid].iloc[0]
        print(f'  {bid}: ctID={ref_row["coverageTypeID"]} desc={str(ref_row["description"])[:60]}')
    print()
    print(f'In our output but NOT in reference: {len(extra)}')
    for bid in extra:
        print(f'  {bid}')
else:
    print(f'No reference file at {REF_PATH} - skipping comparison')


## 7. Send the output

Final CSV is at the path above. Ship to Morteza when satisfied.